# Our First Class and Object (12/10/2026)

In [10]:
class PartyAnimal:

    def __init__(self):
        self.x = 0

    def party(self):
        self.x = self.x + 1
        print("So far", self.x)

In [11]:
an = PartyAnimal() #Se construye el objeto/instancia

In [12]:
an.party() #1 vez que se usa el metodo .party
an.party() #2 vez que se usa el metodo .party
an.party() #3 vez que se usa el metodo .party

So far 1
So far 2
So far 3


In [13]:
print("Type", type(an))

Type <class '__main__.PartyAnimal'>


In [14]:
print("Dir", dir(an))

Dir ['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'party', 'x']


In [15]:
print("Type", type(an.x))

Type <class 'int'>


In [16]:
print("Type", type(an.party))

Type <class 'method'>


# Object Life Cycle (13/10/2026)

In [17]:
class PartyAnimal:

    def __init__(self):
        self.x = 0
        print('I am constructed')

    def party(self):
        self.x = self.x + 1
        print("So far", self.x)
    
    def __del__(self):
        print('I am destructed', self.x)

In [18]:
an = PartyAnimal() #Se construye el objeto/instancia

I am constructed


In [19]:
an.party() #1 vez que se usa el metodo .party
an.party() #2 vez que se usa el metodo .party

So far 1
So far 2


In [20]:
an = 42 #Se destruye objeto/instancia

I am destructed 2


In [21]:
print('an contains', an)

an contains 42


Una misma clase puede alvergar varios objetos o instancias

In [22]:
class PartyAnimal:

    def __init__(self, z):
        self.x = 0
        self.name = z
        print(self.name,'constructed')

    def party(self):
        self.x = self.x + 1
        print(self.name,"party count", self.x)

In [23]:
s = PartyAnimal("Sally")

Sally constructed


In [24]:
s.party()

Sally party count 1


In [25]:
j = PartyAnimal("Jim")

Jim constructed


In [26]:
j.party()
s.party()

Jim party count 1
Sally party count 2


# Object Inheritance (13/10/2026)

Clase padre

In [27]:
class PartyAnimal:

    def __init__(self, nam):
        self.x = 0
        self.name = nam
        print(self.name, "constructed")

    def party(self):
        self.x = self.x + 1
        print(self.name, "party count", self.x)

Clase hija (hereda)

In [28]:
class FootballFan(PartyAnimal):

    def __init__(self, nam):
        super().__init__(nam) #Inicializa PartyAnimal
        self.points = 0

    def touchdown(self):
        self.points = self.points + 7
        self.party()
        print(self.name, "points", self.points)

super().__init__(nam)
hace lo siguiente:
1. super() permite acceder a los métodos de la clase padre siguiendo el orden de resolución de métodos de Python.
2. .__init__(nam) llama al constructor de esa clase, pasándole el nombre recibido.
3. El constructor inicializa self.x y self.name, e imprime el mensaje correspondiente.
4. Finalmente, self.points = 0 inicializa el atributo adicional de FootballFan.

In [29]:
s = PartyAnimal("Sally")
s.party()

Sally constructed
Sally party count 1


In [30]:
j = FootballFan("Jim")
j.party()
j.touchdown()

Jim constructed
Jim party count 1
Jim party count 2
Jim points 7


# Worked Example: Counting Email in a Database (13/10/2026)

In [2]:
import sqlite3

# Conectar con la base de datos.
# Si emaildb.sqlite no existe, SQLite la crea automáticamente.
conn = sqlite3.connect('emaildb.sqlite')

# Crear un cursor para ejecutar comandos SQL.
cur = conn.cursor()

# Eliminar la tabla Counts si ya existe.
# Esto permite empezar el conteo desde cero en cada ejecución.
cur.execute('DROP TABLE IF EXISTS Counts')

# Crear la tabla Counts con dos columnas:
# email: almacena el correo electrónico como texto.
# count: almacena el número de apariciones como entero.
cur.execute('''
CREATE TABLE Counts (email TEXT, count INTEGER)
''')

# Pedir al usuario el nombre del archivo que contiene los correos.
fname = input('Enter file name: ')

# Si el usuario no escribe ningún nombre, utilizar el archivo por defecto.
if len(fname) < 1:
    fname = 'mbox-short.txt'

# Abrir el archivo de texto para leerlo.
fh = open(fname)

# Recorrer el archivo línea por línea.
for line in fh:

    # Ignorar las líneas que no comienzan con 'From '.
    # Estas líneas no se utilizan para contar los correos.
    if not line.startswith('From: '):
        continue

    # Separar la línea en palabras usando los espacios.
    pieces = line.split()

    # El correo electrónico está en la segunda posición (índice 1).
    email = pieces[1]

    # Buscar si ese correo ya existe en la tabla Counts.
    # El signo ? es un marcador para pasar el valor de forma segura.
    cur.execute(
        'SELECT count FROM Counts WHERE email = ?',
        (email,)
    )

    # Obtener el primer resultado de la consulta.
    row = cur.fetchone()

    # Si no existe ningún registro para ese correo...
    if row is None:

        # Insertar el correo por primera vez, con un conteo de 1.
        cur.execute('''
            INSERT INTO Counts (email, count)
            VALUES (?, 1)
        ''', (email,))

    else:

        # Si el correo ya existe, incrementar su contador en 1.
        cur.execute('''
            UPDATE Counts
            SET count = count + 1
            WHERE email = ?
        ''', (email,))
        
# Guardar los cambios realizados en la base de datos.
conn.commit()

# Definir una consulta para obtener los 10 correos más frecuentes.
# ORDER BY count DESC ordena de mayor a menor frecuencia.
# LIMIT 10 devuelve como máximo 10 registros.
sqlstr = '''
SELECT email, count
FROM Counts
ORDER BY count DESC
LIMIT 10
'''

# Ejecutar la consulta y recorrer los resultados.
for row in cur.execute(sqlstr):

    # Mostrar el correo y su número de apariciones.
    print(str(row[0]), row[1])

# Cerrar el archivo y la conexión a la base de datos.
fh.close()
conn.close()

Enter file name: 
zqian@umich.edu 195
mmmay@indiana.edu 161
cwen@iupui.edu 158
chmaurer@iupui.edu 111
aaronz@vt.edu 110
ian@caret.cam.ac.uk 96
jimeng@umich.edu 93
rjlowe@iupui.edu 90
dlhaines@umich.edu 84
david.horwitz@uct.ac.za 67


# Homework: Counting Organizations (13/10/2026)

In [1]:
import sqlite3

# Conectarse a la base de datos emaildb2.sqlite.
# Si el archivo no existe, SQLite lo crea automáticamente.
conn = sqlite3.connect('emaildb2.sqlite')

# Crear un cursor para ejecutar comandos SQL.
cur = conn.cursor()

# Eliminar la tabla Counts si ya existe.
# Esto permite comenzar el conteo desde cero.
cur.execute('''
DROP TABLE IF EXISTS Counts''')

# Crear la tabla Counts con dos columnas:
# org: almacena el dominio de la organización.
# count: almacena el número de mensajes de ese dominio.
cur.execute('''
CREATE TABLE Counts (org TEXT, count INTEGER)''')

# Solicitar al usuario el nombre del archivo que contiene los correos.
fname = input('Enter file name: ')

# Si el usuario no escribe ningún nombre, utilizar mbox.txt.
if (len(fname) < 1):
    fname = 'mbox.txt'

# Abrir el archivo para leerlo.
fh = open(fname)

# Recorrer el archivo línea por línea.
for line in fh:

    # Ignorar las líneas que no comienzan con 'From: '.
    # continue hace que Python pase a la siguiente línea.
    if not line.startswith('From: '):
        continue

    # Dividir la línea en palabras usando los espacios.
    pieces = line.split()

    # Obtener el correo electrónico, que está en la segunda posición.
    email = pieces[1]

    # Encontrar la posición del símbolo @ en el correo electrónico.
    # Por ejemplo, en 'ana@gmail.com', find('@') devuelve 3.
    dom = email.find('@')

    # Extraer todo lo que aparece después del símbolo @.
    # Por ejemplo, 'ana@gmail.com' se convierte en 'gmail.com'.
    org = email[dom + 1:len(email)]

    # Buscar en la tabla Counts si el dominio ya está registrado.
    # El signo ? es un marcador para el valor que se pasa en (org,).
    cur.execute('SELECT count FROM Counts WHERE org = ? ', (org,))

    # Recuperar la primera fila del resultado.
    # Si el dominio no existe en la tabla, row será None.
    row = cur.fetchone()

    # Si no existe ningún registro para ese dominio...
    if row is None:

        # Insertar el dominio en la tabla con un contador inicial de 1.
        cur.execute('''INSERT INTO Counts (org, count)
                VALUES (?, 1)''', (org,))

    else:

        # Si el dominio ya existe, incrementar su contador en 1.
        # WHERE org = ? garantiza que solo se actualice ese dominio.
        cur.execute('UPDATE Counts SET count = count + 1 WHERE org = ?',
                    (org,))

# Guardar los cambios realizados en la base de datos.
conn.commit()

# Consulta SQL para seleccionar las 10 organizaciones más frecuentes.
# ORDER BY count DESC ordena de mayor a menor.
# LIMIT 10 devuelve como máximo 10 filas.
# Referencia: https://www.sqlite.org/lang_select.html
sqlstr = 'SELECT org, count FROM Counts ORDER BY count DESC LIMIT 10'

# Ejecutar la consulta y recorrer los resultados.
for row in cur.execute(sqlstr):

    # Imprimir el dominio de la organización y su número de apariciones.
    # row[0] contiene el dominio y row[1] contiene el contador.
    print(str(row[0]), row[1])

# Cerrar el archivo, el cursor y la conexión con la base de datos.
fh.close()
cur.close()
conn.close()

Enter file name: 
iupui.edu 536
umich.edu 491
indiana.edu 178
caret.cam.ac.uk 157
vt.edu 110
uct.ac.za 96
media.berkeley.edu 56
ufp.pt 28
gmail.com 25
et.gatech.edu 17


# Homework: Tracks (14/10/2026)

In [1]:
import sqlite3

# Conecta con la base de datos SQLite llamada 'trackdb.sqlite'.
# Si el archivo no existe, Python lo crea automáticamente.
conn = sqlite3.connect('trackdb.sqlite')

# Crea un cursor para ejecutar instrucciones SQL sobre la base de datos.
cur = conn.cursor()


# ============================================================
# 1. ELIMINAR LAS TABLAS SI YA EXISTEN
# ============================================================

# Elimina la tabla Artist si existe.
# Esto permite comenzar desde cero al ejecutar el programa.
cur.execute('''
DROP TABLE IF EXISTS Artist''')

# Elimina la tabla Genre si existe.
cur.execute('''
DROP TABLE IF EXISTS Genre''')

# Elimina la tabla Album si existe.
cur.execute('''
DROP TABLE IF EXISTS Album''')

# Elimina la tabla Track si existe.
cur.execute('''
DROP TABLE IF EXISTS Track''')


# ============================================================
# 2. CREAR LAS TABLAS DE LA BASE DE DATOS
# ============================================================

# Crea la tabla Artist para almacenar los artistas musicales.
cur.execute('''
CREATE TABLE Artist (
    id  INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    name    TEXT UNIQUE
)''')

# Crea la tabla Genre para almacenar los géneros musicales.
cur.execute('''
CREATE TABLE Genre (
    id  INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    name    TEXT UNIQUE
)''')

# Crea la tabla Album para almacenar los álbumes.
# artist_id identifica al artista asociado con cada álbum.
cur.execute('''
CREATE TABLE Album (
    id  INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    artist_id  INTEGER,
    title   TEXT UNIQUE
)''')

# Crea la tabla Track para almacenar las canciones.
# album_id identifica el álbum de cada canción.
# genre_id identifica el género musical de cada canción.
# len almacena la duración, rating la calificación y count el número de reproducciones.
cur.execute('''
CREATE TABLE Track (
    id  INTEGER NOT NULL PRIMARY KEY 
        AUTOINCREMENT UNIQUE,
    title TEXT  UNIQUE,
    album_id  INTEGER,
    genre_id  INTEGER,
    len INTEGER, 
    rating INTEGER, 
    count INTEGER
)''')


# ============================================================
# 3. SOLICITAR EL NOMBRE DEL ARCHIVO
# ============================================================

# Pide al usuario el nombre del archivo que contiene los datos.
fname = input('Enter file name: ')

# Si el usuario no escribe ningún nombre, utiliza 'tracks.csv'.
if (len(fname) < 1):
    fname = 'tracks.csv'

# Abre el archivo para leer su contenido.
fh = open(fname)


# ============================================================
# 4. LEER Y PROCESAR CADA LÍNEA DEL ARCHIVO
# ============================================================

# Ejemplo de una línea del archivo:
# Another One Bites The Dust,Queen,Greatest Hits,55,100,217103,Rock
# 0                          1     2             3  4   5      6

# Recorre el archivo línea por línea.
for line in fh:

    # Elimina espacios y saltos de línea al principio y al final.
    line = line.strip()

    # Divide la línea en una lista usando la coma como separador.
    pieces = line.split(',')

    # Si la línea tiene menos de 7 elementos, la ignora
    # y pasa a la siguiente iteración del ciclo.
    if len(pieces) < 7:
        continue

    # Guarda cada elemento de la lista en una variable.
    track = pieces[0]   # Nombre de la canción.
    artist = pieces[1]  # Nombre del artista.
    album = pieces[2]   # Nombre del álbum.
    count = pieces[3]   # Número de reproducciones.
    rating = pieces[4]  # Calificación de la canción.
    length = pieces[5]  # Duración de la canción.
    genre = pieces[6]   # Género musical.

    # Esta instrucción, si se descomenta, muestra los datos leídos.
    # print(track, artist, album, count, rating, length, genre)


    # ========================================================
    # 5. INSERTAR EL ARTISTA Y OBTENER SU ID
    # ========================================================

    # Inserta el artista si todavía no existe.
    # INSERT OR IGNORE evita insertar un duplicado si el nombre
    # ya existe, porque la columna name es UNIQUE.
    cur.execute('''INSERT OR IGNORE INTO Artist (name)
        VALUES ( ? )''', (artist, ))

    # Busca el ID del artista usando su nombre.
    # El signo ? se reemplaza de forma segura por el valor artist.
    cur.execute('SELECT id FROM Artist WHERE name = ? ', (artist, ))

    # Recupera la fila del resultado y extrae su primer valor: id.
    artist_id = cur.fetchone()[0]


    # ========================================================
    # 6. INSERTAR EL GÉNERO Y OBTENER SU ID
    # ========================================================

    # Inserta el género si todavía no existe.
    cur.execute('''INSERT OR IGNORE INTO Genre (name)
        VALUES ( ? )''', (genre, ))

    # Busca el ID correspondiente al nombre del género.
    cur.execute('SELECT id FROM Genre WHERE name = ? ', (genre, ))

    # Obtiene el ID del género encontrado.
    genre_id = cur.fetchone()[0]


    # ========================================================
    # 7. INSERTAR EL ÁLBUM Y OBTENER SU ID
    # ========================================================

    # Inserta el álbum junto con el ID de su artista,
    # siempre que el título del álbum no exista ya.
    cur.execute('''INSERT OR IGNORE INTO Album (artist_id, title)
        VALUES ( ?, ?)''', (artist_id, album))

    # Busca el ID del álbum mediante su título.
    cur.execute('SELECT id FROM Album WHERE title = ? ', (album, ))

    # Obtiene el ID del álbum encontrado.
    album_id = cur.fetchone()[0]


    # ========================================================
    # 8. INSERTAR LA CANCIÓN EN LA TABLA TRACK
    # ========================================================

    # Inserta la canción con sus datos y los ID asociados.
    # INSERT OR REPLACE reemplaza una fila existente si ocurre
    # un conflicto con una restricción UNIQUE, por ejemplo,
    # si ya existe una canción con el mismo título.
    cur.execute('''INSERT OR REPLACE INTO Track
        (title, album_id, genre_id, len, rating, count)
        VALUES ( ?, ?, ?, ?, ?, ?)''',
        (track, album_id, genre_id, length, rating, count))


# ============================================================
# 9. GUARDAR LOS CAMBIOS
# ============================================================

# Confirma los cambios realizados en la base de datos.
# Sin esta confirmación, los cambios podrían no quedar guardados.
conn.commit()

# Cierra el archivo después de terminar de leerlo.
fh.close()

# Cierra la conexión con la base de datos.
conn.close()

Enter file name: 


In [3]:
import sqlite3

# ============================================================
# 1. CONECTARSE A LA BASE DE DATOS
# ============================================================

# Abre una conexión con la base de datos llamada 'trackdb.sqlite'.
# Si el archivo no existe, SQLite lo crea automáticamente.
conn = sqlite3.connect('trackdb.sqlite')

# Crea un cursor para ejecutar consultas SQL en la base de datos.
cur = conn.cursor()


# ============================================================
# 2. EJECUTAR LA CONSULTA SQL
# ============================================================

# Selecciona cuatro columnas:
# - Track.title: nombre de la canción.
# - Artist.name: nombre del artista.
# - Album.title: nombre del álbum.
# - Genre.name: nombre del género musical.
#
# FROM Track indica que la consulta comienza en la tabla Track.
#
# JOIN Genre relaciona cada canción con su género musical,
# utilizando Track.genre_id = Genre.id.
#
# JOIN Album relaciona cada canción con su álbum,
# utilizando Track.album_id = Album.id.
#
# JOIN Artist relaciona cada álbum con su artista,
# utilizando Album.artist_id = Artist.id.
#
# ORDER BY Artist.name ordena los resultados alfabéticamente
# según el nombre del artista.
#
# LIMIT 3 limita el resultado a un máximo de tres filas.

cur.execute('''
    SELECT Track.title, Artist.name, Album.title, Genre.name
    FROM Track
    JOIN Genre ON Track.genre_id = Genre.id
    JOIN Album ON Track.album_id = Album.id
    JOIN Artist ON Album.artist_id = Artist.id
    ORDER BY Artist.name
    LIMIT 3
''')


# ============================================================
# 3. RECUPERAR LOS RESULTADOS
# ============================================================

# Obtiene todas las filas devueltas por la consulta SQL.
# Como utilizamos LIMIT 3, se recuperan como máximo tres filas.
rows = cur.fetchall()


# ============================================================
# 4. MOSTRAR LOS RESULTADOS
# ============================================================

# Recorre cada fila de los resultados.
for row in rows:

    # Imprime los datos de cada canción.
    # Cada fila es una tupla con cuatro valores:
    # título de la canción, artista, álbum y género.
    print(row)


# ============================================================
# 5. CERRAR LA CONEXIÓN
# ============================================================

# Cierra la conexión con la base de datos.
conn.close()

('For Those About To Rock (We Salute You)', 'AC/DC', 'Who Made Who', 'Rock')
('Hells Bells', 'AC/DC', 'Who Made Who', 'Rock')
('Shake Your Foundations', 'AC/DC', 'Who Made Who', 'Rock')


# Homework: Many Students in Many Courses (14/10/2026)

In [1]:
import json
import sqlite3

# ============================================================
# 1. CONECTARSE A LA BASE DE DATOS
# ============================================================

# Abre una conexión con la base de datos rosterdb.sqlite.
# Si el archivo no existe, SQLite lo crea automáticamente.
conn = sqlite3.connect('rosterdb.sqlite')

# Crea un cursor para ejecutar instrucciones SQL.
cur = conn.cursor()


# ============================================================
# 2. PREPARAR LA BASE DE DATOS
# ============================================================

# executescript() permite ejecutar varias instrucciones SQL
# dentro de una misma cadena de texto.

cur.executescript('''
DROP TABLE IF EXISTS User;
DROP TABLE IF EXISTS Member;
DROP TABLE IF EXISTS Course;

CREATE TABLE User (
    id     INTEGER PRIMARY KEY,
    name   TEXT UNIQUE
);

CREATE TABLE Course (
    id     INTEGER PRIMARY KEY,
    title  TEXT UNIQUE
);

CREATE TABLE Member (
    user_id     INTEGER,
    course_id   INTEGER,
    role        INTEGER,
    PRIMARY KEY (user_id, course_id)
)
''')


# ============================================================
# 3. SOLICITAR Y LEER EL ARCHIVO JSON
# ============================================================

# Pide al usuario el nombre del archivo que contiene los datos.
fname = input('Enter file name: ')

# Si el usuario no escribe ningún nombre, utiliza
# el archivo roster_data.json.
if len(fname) < 1:
    fname = 'roster_data.json'

# Abre el archivo, lee todo su contenido y lo guarda como texto.
str_data = open(fname).read()

# Convierte el texto JSON en una estructura de datos de Python,
# normalmente una lista de listas en este ejercicio.
json_data = json.loads(str_data)

# Ejemplo de los datos contenidos en el archivo JSON:
# [
#     ["Charley", "si110", 1],
#     ["Mea", "si110", 0]
# ]
#
# Cada elemento contiene:
# [nombre del usuario, título del curso, rol]


# ============================================================
# 4. RECORRER LOS DATOS DEL ARCHIVO JSON
# ============================================================

# Recorre cada registro de la lista json_data.
for entry in json_data:

    # Extrae los datos de cada registro según su posición.
    name = entry[0]   # Nombre del usuario.
    title = entry[1]  # Título del curso.
    role = entry[2]   # Rol del usuario en ese curso.

    # Esta instrucción, si se descomenta, muestra los datos.
    # print((name, title, role))


    # ========================================================
    # 5. INSERTAR EL USUARIO Y OBTENER SU ID
    # ========================================================

    # Inserta el usuario si su nombre todavía no existe.
    # INSERT OR IGNORE evita errores por nombres duplicados,
    # debido a la restricción UNIQUE de la columna name.
    cur.execute('''INSERT OR IGNORE INTO User (name)
        VALUES ( ? )''', (name, ))

    # Busca el ID del usuario mediante su nombre.
    # El signo ? se reemplaza por el valor de name.
    cur.execute('SELECT id FROM User WHERE name = ? ', (name, ))

    # Recupera la fila del resultado y obtiene su primer valor:
    # el ID del usuario.
    user_id = cur.fetchone()[0]


    # ========================================================
    # 6. INSERTAR EL CURSO Y OBTENER SU ID
    # ========================================================

    # Inserta el curso si su título todavía no existe.
    # INSERT OR IGNORE evita insertar títulos duplicados.
    cur.execute('''INSERT OR IGNORE INTO Course (title)
        VALUES ( ? )''', (title, ))

    # Busca el ID del curso mediante su título.
    cur.execute('SELECT id FROM Course WHERE title = ? ', (title, ))

    # Recupera el ID del curso encontrado.
    course_id = cur.fetchone()[0]


    # ========================================================
    # 7. REGISTRAR LA RELACIÓN ENTRE USUARIO Y CURSO
    # ========================================================

    # Inserta o reemplaza la relación entre el usuario y el curso.
    # Se almacenan el ID del usuario, el ID del curso y su rol.
    #
    # INSERT OR REPLACE reemplaza el registro existente si ya
    # hay una fila con la misma clave primaria compuesta:
    # (user_id, course_id).
    cur.execute('''INSERT OR REPLACE INTO Member
        (user_id, course_id, role) VALUES ( ?, ?, ? )''',
        (user_id, course_id, role))


# ============================================================
# 8. GUARDAR LOS CAMBIOS
# ============================================================

# Confirma y guarda todas las modificaciones realizadas
# en la base de datos.
conn.commit()

# Cierra la conexión con la base de datos.
conn.close()

Enter file name: 


In [2]:
import sqlite3

# 1. Conectarse a la base de datos
conn = sqlite3.connect('rosterdb.sqlite')
cur = conn.cursor()

# 2. Ejecutar la consulta SQL
cur.execute('''
    SELECT User.name, Course.title, Member.role
    FROM User
    JOIN Member ON User.id = Member.user_id
    JOIN Course ON Member.course_id = Course.id
    ORDER BY User.name DESC,
             Course.title DESC,
             Member.role DESC
    LIMIT 2
''')

# 3. Recuperar los resultados
rows = cur.fetchall()

# 4. Mostrar los resultados
print("Usuario | Curso | Rol")
print("-" * 50)

for row in rows:
    print(row)

# 5. Cerrar la conexión
conn.close()

Usuario | Curso | Rol
--------------------------------------------------
('Zunera', 'si106', 0)
('Zulaikha', 'si422', 0)


In [3]:
import sqlite3

# 1. Conectarse a la base de datos
conn = sqlite3.connect('rosterdb.sqlite')
cur = conn.cursor()

# 2. Ejecutar la consulta SQL
cur.execute('''
    SELECT 'XYZZY' || hex(User.name || Course.title || Member.role ) AS X FROM 
    User JOIN Member JOIN Course 
    ON User.id = Member.user_id AND Member.course_id = Course.id
    ORDER BY X LIMIT 1;
''')

# 3. Recuperar los resultados
print(cur.fetchall())

# 4. Cerrar la conexión
conn.close()

[('XYZZY4161697661736934333030',)]
